# Chandrayaan-2 Multi-Modal Image Correspondence Backend

This notebook runs the FastAPI backend on Google Colab with GPU support.

**Steps:**
1. Clone the repo
2. Install dependencies
3. Start the backend server
4. Expose via ngrok

## Step 1: Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/1sarthak7/SIH-2026.git"
REPO_DIR = "/content/SIH-2026"

if os.path.exists(REPO_DIR):
    print("Repo already cloned, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

print(f"\nRepo ready at {REPO_DIR}")
!ls {REPO_DIR}/backend/app/services/

## Step 2: Install Dependencies

In [ ]:
# Install system deps for GDAL/rasterio
!apt-get update -qq && apt-get install -y -qq libgdal-dev gdal-bin > /dev/null 2>&1
print("System deps installed.")

# Install Python packages
!pip install -q fastapi uvicorn[standard] python-multipart pydantic pydantic-settings loguru aiofiles httpx tqdm
!pip install -q kornia opencv-python-headless scikit-learn scikit-image
!pip install -q rasterio pyproj shapely pandas
!pip install -q pyngrok

print("\nAll Python deps installed.")

# Verify key imports
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Upload Data (Optional)

If you have ISRO Chandrayaan-2 `.img` or `.tif` files, upload them to `/content/SIH-2026/Data/`.

In [ ]:
DATA_DIR = "/content/SIH-2026/Data"
os.makedirs(DATA_DIR, exist_ok=True)

# Set environment variable so the backend knows where data lives
os.environ["DATA_DIR"] = DATA_DIR

# List existing data files if any
data_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        data_files.append(os.path.join(root, f))

if data_files:
    print(f"Found {len(data_files)} data files:")
    for f in data_files[:10]:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")
else:
    print("No data files found. Upload .img/.tif files to proceed.")
    print("You can still test with the /health endpoint and demo data.")

## Step 4: Start Backend Server

In [ ]:
import subprocess
import time

BACKEND_DIR = "/content/SIH-2026/backend"

# Verify the backend directory exists
assert os.path.exists(BACKEND_DIR), f"Backend not found at {BACKEND_DIR}. Re-run Step 1."
assert os.path.exists(f"{BACKEND_DIR}/app/main.py"), "app/main.py not found!"

print("Starting FastAPI server...")

# Start uvicorn as a subprocess
proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=BACKEND_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Wait for startup and print first lines
time.sleep(5)

# Read initial output
import select
while select.select([proc.stdout], [], [], 0.1)[0]:
    line = proc.stdout.readline()
    if line:
        print(line, end="")

# Check if process is still running
if proc.poll() is None:
    print("\nServer is running on port 8000")
else:
    print(f"\nServer exited with code {proc.returncode}")
    # Print remaining output for debugging
    remaining = proc.stdout.read()
    if remaining:
        print(remaining)

## Step 5: Expose with ngrok

In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token (get one at https://dashboard.ngrok.com)
NGROK_TOKEN = ""  # <-- Paste your token here

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Open tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"  Backend is live at: {public_url}")
print(f"  API Docs:          {public_url}/docs")
print(f"  Health Check:      {public_url}/health")
print(f"{'='*60}")
print(f"\nSet this URL in your frontend .env.local:")
print(f"  NEXT_PUBLIC_API_URL={public_url}")

## Step 6: Test Health Check

In [ ]:
import requests

resp = requests.get("http://localhost:8000/health")
print("Health check response:")
print(resp.json())

## Step 7: Monitor Server Logs

Run this cell to see live server output (stop manually when done).

In [ ]:
# Stream server logs (Ctrl+C or stop cell to quit)
try:
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(0.5)
except KeyboardInterrupt:
    print("\nStopped log streaming.")

if proc.poll() is not None:
    print(f"Server process exited with code {proc.returncode}")

## Cleanup

In [ ]:
# Kill the server and close ngrok tunnel
proc.terminate()
ngrok.kill()
print("Server stopped, tunnel closed.")